# 07 - Explainability

**Architecture block 6.** Grad-CAM, SHAP, integrated gradients, graph
explainability and attention maps.

Requires a trained model: run `scripts/04_train_and_ablate.py` first.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import cropforecast
from cropforecast.config import load_config, ensure_dirs, set_seed, Device
cfg = load_config(Path.cwd().parent / "configs" / "default.yaml")
ensure_dirs(cfg); set_seed(cfg.project.seed)
device = Device.auto(cfg.training.amp)
print("device:", device)

In [ ]:
import torch, pandas as pd, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from cropforecast.models.full_model import CropDiseaseForecastNet
from cropforecast.models.backbones import load_backbone, SPECS
from transformers import AutoImageProcessor

blob = torch.load(Path(cfg.paths.checkpoints) / "final_model.pt", map_location=device.name)
m, c = blob["meta"], blob["config"]
model = CropDiseaseForecastNet(
    vision_dim=m["vision_dim"], climate_dim=m["climate_dim"], meta_dim=m["meta_dim"],
    num_classes=m["num_classes"], horizons=tuple(m["horizons"]),
    fusion_dim=c["fusion_dim"], gnn_conv=c["gnn_conv"], gnn_hidden=c["gnn_hidden"],
    gnn_layers=c["gnn_layers"], gnn_heads=c["gnn_heads"]).to(device.name)
model.load_state_dict(blob["state_dict"]); model.eval()
print("loaded model trained on backbone:", c["backbone"])

## Visual explanations

In [ ]:
from cropforecast.explain.vision import attention_rollout, grad_cam, overlay
bb = load_backbone(c["backbone"], frozen=True, output_attentions=True).to(device.name).eval()
proc = AutoImageProcessor.from_pretrained(SPECS[c["backbone"]].hf_id)
mean = torch.tensor(proc.image_mean, device=device.name).view(1,3,1,1)
std  = torch.tensor(proc.image_std,  device=device.name).view(1,3,1,1)

obs = pd.read_parquet(Path(cfg.paths.processed) / "observations.parquet")
row = obs[obs.class_name == "Tomato___Late_blight"].iloc[0]
img = Image.open(row.image_path).convert("RGB").resize((224,224))
arr = np.array(img, dtype=np.uint8)
x = torch.from_numpy(arr).permute(2,0,1)[None].to(device.name).float()/255
x = (x-mean)/std

roll = attention_rollout(bb, x)
fig, ax = plt.subplots(1,2, figsize=(9,4.5))
ax[0].imshow(arr); ax[0].set_title(row.class_name); ax[0].axis("off")
ax[1].imshow(overlay(arr, roll)); ax[1].set_title("attention rollout"); ax[1].axis("off")
plt.show()

## Integrated gradients over climate

In [ ]:
ig_path = Path(cfg.paths.reports) / "stage5_integrated_gradients.csv"
if ig_path.exists():
    ig = pd.read_csv(ig_path).head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8,6))
    ax.barh(ig.feature, ig.attribution,
            color=["#f43f5e" if v<0 else "#22c55e" for v in ig.attribution])
    ax.axvline(0, color="#888", lw=.8); ax.set_xlabel("attribution to +1d risk")
    plt.tight_layout(); plt.show()
else:
    print("Run scripts/05_explain.py")

## Graph attribution

Which neighbouring farms drive one field's forecast?

In [ ]:
ga = Path(cfg.paths.reports) / "stage5_graph_attribution.csv"
pd.read_csv(ga) if ga.exists() else print("Run scripts/05_explain.py")

## SHAP global importance

In [ ]:
sh = Path(cfg.paths.reports) / "stage5_shap.csv"
pd.read_csv(sh).head(15) if sh.exists() else print("Run scripts/05_explain.py")